# Coin Toss Experiment

## The question

Is a coin toss truly random, or is it deterministic chaos?

We drop a rigid body coin from height $h$ with an initial tilt angle $\theta$ about a chosen axis.
The coin falls under gravity, bounces off a rigid floor (with restitution), and eventually
settles with one face up.

**Key prediction:** At low heights, the outcome (heads vs tails) is almost entirely
determined by the initial tilt — the coin doesn't have time to flip.
As height increases, the coin tumbles and bounces more, and the boundary between
heads and tails in the (height, angle) plane becomes increasingly complex — a
signature of deterministic chaos.

## The model

- **Coin**: rigid body disk (mass $m$, radius $r$, thin thickness)
- **Inertia tensor**: $I_\text{diam} = mr^2/4$, $I_\text{axis} = mr^2/2$
- **Floor**: rigid constraint at $y = 0$ with coefficient of restitution $e$
- **Integrator**: symplectic Leapfrog with contact impulse for angular + linear momentum
- **Outcome**: determined by which direction the coin's normal vector points after settling

## Convention

- **Heads** ($+1$, blue): coin normal points up ($\hat{y}_{\text{body}} \cdot \hat{y}_{\text{world}} > 0$)
- **Tails** ($-1$, red): coin normal points down
- **Edge** ($0$, white): coin balanced on rim (rare)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from lab.experiments.coin_toss import (
    toss_coin, sweep_h_vs_angle, plot_outcome_map,
    plot_single_toss, classify_final_orientation,
    HEADS, TAILS, EDGE,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Single coin tosses — visualising the trajectory

Let's start by watching a few individual tosses at different tilt angles.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=False)

test_angles = [0, 30, 60, 100, 150, 180]
height = 0.5

for ax, angle_deg in zip(axes.flat, test_angles):
    angle = np.radians(angle_deg)
    result, traj = toss_coin(
        height=height, tilt_axis='x', tilt_angle=angle,
        restitution=0.5, dt=0.001, record_trajectory=True
    )
    plot_single_toss(traj, result, ax=ax)
    label = {HEADS: 'Heads', TAILS: 'Tails', EDGE: 'Edge'}[result]
    ax.set_title(f'θ = {angle_deg}° → {label}')

fig.suptitle(f'Single tosses from h = {height} m', fontsize=14)
fig.tight_layout()
plt.show()

## 2. The deterministic regime — low height

At low heights, the coin barely has time to rotate before hitting the floor.
The outcome is determined almost entirely by the initial tilt angle.

Let's sweep tilt angle at a fixed low height and see the sharp transition.

In [ ]:
angles_fine = np.linspace(0, np.pi, 60)
results_low = []

h_low = 0.15
for angle in angles_fine:
    results_low.append(toss_coin(height=h_low, tilt_axis='x', tilt_angle=angle,
                                  restitution=0.5, dt=0.001))

results_low = np.array(results_low)

fig, ax = plt.subplots(figsize=(10, 3))
colors = ['#d62728' if r == TAILS else '#1f77b4' if r == HEADS else 'gray'
          for r in results_low]
ax.bar(np.degrees(angles_fine), results_low, width=3.2, color=colors)
ax.set_xlabel('initial tilt about x-axis (degrees)')
ax.set_ylabel('outcome')
ax.set_yticks([-1, 0, 1])
ax.set_yticklabels(['Tails', 'Edge', 'Heads'])
ax.set_title(f'Coin toss outcomes at h = {h_low} m (deterministic regime)')
ax.axhline(0, color='k', lw=0.3)
plt.tight_layout()
plt.show()

n_heads = np.sum(results_low == HEADS)
n_tails = np.sum(results_low == TAILS)
print(f'At h = {h_low} m:  {n_heads} heads, {n_tails} tails out of {len(results_low)} trials')
print(f'Sharp boundary near θ ≈ 90° — almost perfectly deterministic')

## 3. The outcome map — height × tilt angle

Now the main result: sweep both height and tilt angle, plot the outcome as a
2D colour map.

- **Blue** = Heads
- **Red** = Tails

At low heights, you should see a clean vertical boundary near 90°.
As height increases, the boundary fractures into complex patterns —
the hallmark of deterministic chaos.

> **Note:** This sweep runs ~300 simulations and takes a few minutes.

In [ ]:
# --- Configuration ---
# Change tilt_axis to 'x', 'y', or 'z' to explore different rotations.
# 'x' and 'z' tilt the coin so it can flip; 'y' spins around the normal.
TILT_AXIS = 'x'

heights = np.linspace(0.1, 3.0, 12)
angles = np.linspace(0, np.pi, 30)

print(f'Running {len(heights)}×{len(angles)} = {len(heights)*len(angles)} simulations...')
results = sweep_h_vs_angle(
    heights, angles,
    tilt_axis=TILT_AXIS,
    restitution=0.5,
    dt=0.001,
)
print('Done!')

In [ ]:
fig, ax = plot_outcome_map(heights, angles, results, tilt_axis=TILT_AXIS)
plt.show()

## 4. Probability vs height

For each height, compute the fraction of angles that give heads.
At low heights this should be close to 50% (half the angles give heads,
half give tails). As height increases, the probability should oscillate
but remain near 50% overall — the system becomes increasingly unpredictable
from the tilt angle alone.

In [ ]:
heads_fraction = np.mean(results == HEADS, axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(heights, heads_fraction, 'o-', color='#1f77b4', lw=2)
ax.axhline(0.5, color='k', ls='--', lw=0.8, label='50%')
ax.set_xlabel('drop height (m)')
ax.set_ylabel('fraction landing heads')
ax.set_title('Heads probability vs drop height')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Comparing tilt axes

The coin is symmetric about its normal (y-axis) but not about the
other axes. Tilting about x vs z should give similar but not identical
maps (the inertia tensor is $I_x = I_z = mr^2/4$, so they're equal).
Tilting about y (the coin's own normal) just spins it — the outcome
should be insensitive to this rotation.

In [ ]:
# Quick comparison: 1D sweep at fixed height for each axis
h_test = 1.0
test_angles = np.linspace(0, np.pi, 40)

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for ax, axis_name in zip(axes, ['x', 'y', 'z']):
    results_1d = np.array([
        toss_coin(height=h_test, tilt_axis=axis_name, tilt_angle=a,
                  restitution=0.5, dt=0.001)
        for a in test_angles
    ])
    colors = ['#d62728' if r == TAILS else '#1f77b4' if r == HEADS else 'gray'
              for r in results_1d]
    ax.bar(np.degrees(test_angles), results_1d, width=4.8, color=colors)
    ax.set_ylabel('outcome')
    ax.set_yticks([-1, 0, 1])
    ax.set_yticklabels(['T', 'E', 'H'])
    ax.set_title(f'tilt axis = {axis_name}')

axes[-1].set_xlabel('tilt angle (degrees)')
fig.suptitle(f'Outcome vs tilt angle at h = {h_test} m — by axis', fontsize=13)
fig.tight_layout()
plt.show()

## 6. Energy dissipation during a toss

The restitution coefficient $e < 1$ means energy is lost on each bounce.
Let's watch the energy decay for a coin that tumbles several times.

In [ ]:
result, traj = toss_coin(
    height=2.0, tilt_axis='x', tilt_angle=np.radians(45),
    restitution=0.5, dt=0.001, record_trajectory=True
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(traj['t'], traj['y'], 'steelblue', lw=1)
ax1.set_ylabel('height (m)')
ax1.set_title(f'Coin drop from 2.0 m, θ=45° — result: {"Heads" if result==HEADS else "Tails"}')
ax1.axhline(0, color='k', lw=0.3)
ax1.grid(True, alpha=0.3)

ax2.plot(traj['t'], traj['energy'], 'coral', lw=1)
ax2.set_xlabel('time (s)')
ax2.set_ylabel('total energy (J)')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Discussion

### What we see

1. **Low heights**: The outcome map shows a clean, nearly vertical boundary
   at $\theta \approx 90°$. Angles below 90° → heads, above 90° → tails.
   This is the *deterministic regime* — the coin has no time to flip.

2. **Increasing heights**: The boundary becomes increasingly jagged and
   complex. The coin has time to tumble, bounce, and transfer between
   translational and rotational modes. Small changes in initial angle
   can flip the outcome.

3. **The y-axis tilt**: Rotating about the coin's normal axis doesn't
   change which face is up — the outcome should be independent of this
   rotation (modulo numerical noise at the bouncing boundary).

### Why this matters

A coin toss is a **deterministic** system — given exact initial conditions,
the outcome is fixed. There is no randomness in the physics. What makes
real coin tosses appear random is:

- **Sensitivity to initial conditions**: at sufficient height, the outcome
  depends on the initial angle to a precision beyond human control.
- **Basin boundaries**: the boundary between heads and tails outcomes in
  parameter space becomes fractal-like, so any finite uncertainty in the
  initial conditions spans both basins.

This is the essence of *deterministic chaos*: the system is fully
predictable in principle, but unpredictable in practice.

### Parameters to explore

- `restitution`: higher values (closer to 1) mean more bouncing → more chaos
- `radius`: larger coins have higher moments of inertia → slower tumbling
- `angular_momentum`: give the coin an initial spin before dropping
- `TILT_AXIS`: try 'z' for a different rotation axis